In [36]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage
from typing import TypedDict

In [45]:
load_dotenv()

True

In [47]:
# Initialize the model (e.g., using Meta's Llama 3)
model = ChatOpenRouter(
    model="cohere/north-mini-code:free"
)



In [48]:
# state

class LLMState(TypedDict):

    question: str
    answer: str

In [49]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke([HumanMessage(content=prompt)]).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [50]:
# create our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile the graph
workflow = graph.compile()

In [ ]:
# execute
initial_state = {'question':'How far is moon from the earth?'}

final_state = workflow.invoke(initial_state)

final_state